In [ ]:
from datasets import load_from_disk

train_path = "data/common_voice_train"
test_path = "data/common_voice_test"

train_dataset = load_from_disk(train_path)
test_dataset = load_from_disk(test_path)


FileNotFoundError: Directory Lab_commonvoice/data/common_voice_train not found

In [2]:
train_dataset

Dataset({
    features: ['path', 'sentence', 'audio', 'sentence_en'],
    num_rows: 4082
})

In [3]:
test_dataset

Dataset({
    features: ['path', 'sentence', 'audio', 'sentence_en'],
    num_rows: 1934
})

In [4]:
sample = train_dataset[0]

In [5]:
for key, value in sample.items():
    print(f"{key}: {value}\n")

path: common_voice_mn_21799896.mp3

sentence: хэмээн өөрийгөө тойрч зогссон шавь нартаа туньсан янзтай уйлж байлаа

audio: {'path': 'common_voice_mn_21799896.mp3', 'array': array([-1.17961196e-16,  2.08166817e-17, -9.02056208e-17, ...,
        2.21929241e-07,  2.48354900e-06, -9.85941142e-07], shape=(103296,)), 'sampling_rate': 16000}

sentence_en: and the disciples were standing around him weeping and wailing



In [6]:
from transformers import WhisperProcessor

outliers = {}

processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="Mongolian", task="transcribe")


for idx, sample in enumerate(train_dataset):

    token_count = len(processor.tokenizer(sample["sentence"]).input_ids)

    if token_count > 448:
        outliers[idx] = sample

print(f"Found {len(outliers)} outliers.")

Found 0 outliers.


In [7]:
clean_train_dataset = train_dataset.filter(
    lambda sample: len(processor.tokenizer(
        sample["sentence"]).input_ids) <= 448
)




Filter:   0%|          | 0/4082 [00:00<?, ? examples/s]

In [8]:
for idx, sample in enumerate(test_dataset):

    token_count = len(processor.tokenizer(sample["sentence"]).input_ids)

    if token_count > 448:
        outliers[idx] = sample

print(f"Found {len(outliers)} outliers.")

Found 0 outliers.


In [ ]:
from datasets import load_from_disk, DatasetDict
from huggingface_hub import login

hf_token = ""
login(token=hf_token)

train_path = "data/common_voice_train"
test_path = "data/common_voice_test"

train_dataset = load_from_disk(train_path)
test_dataset = load_from_disk(test_path)

combined_dataset = DatasetDict({
    "train": train_dataset,
    "validation": test_dataset
})

repo_id = "Ganaa0614/mongolian-stt-translated"

combined_dataset.push_to_hub(repo_id)

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Map:   0%|          | 0/4082 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/41 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...

Map:   0%|          | 0/1934 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/datasets/Ganaa0614/mongolian-stt-translated/commit/d6937b356281bbfa5a97a910ff042c89442161a6', commit_message='Upload dataset', commit_description='', oid='d6937b356281bbfa5a97a910ff042c89442161a6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Ganaa0614/mongolian-stt-translated', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Ganaa0614/mongolian-stt-translated'), pr_revision=None, pr_num=None)

In [7]:
from datasets import load_from_disk, load_dataset
import os 


base_dir = "data/my_voice_dataset_raw"

data_files = {
    "train": os.path.join(base_dir, "train.tsv")
}

data = load_dataset("csv", data_files=data_files, delimiter="\t")

In [8]:
data

DatasetDict({
    train: Dataset({
        features: ['path', 'sentence'],
        num_rows: 100
    })
})

In [8]:
splitted = data["train"].train_test_split(test_size=0.3, seed=42)

In [9]:
splitted["validation"] = splitted.pop("test")

In [10]:
splitted

DatasetDict({
    train: Dataset({
        features: ['path', 'sentence'],
        num_rows: 70
    })
    validation: Dataset({
        features: ['path', 'sentence'],
        num_rows: 30
    })
})

In [11]:
splitted.save_to_disk("data/my_voice_dataset_processed")

Saving the dataset (0/1 shards):   0%|          | 0/70 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

In [23]:
from datasets import load_from_disk, load_dataset

data = load_from_disk("/home/gantumur/Documents/DL/Lab_commonvoice/data/my_voice_dataset_processed")
# data = load_dataset("Ganaa0614/mongolian-custom-stt-translated")

In [24]:
data

DatasetDict({
    train: Dataset({
        features: ['path', 'sentence', 'audio', 'sentence_en'],
        num_rows: 70
    })
    validation: Dataset({
        features: ['path', 'sentence', 'audio', 'sentence_en'],
        num_rows: 30
    })
})

In [ ]:
# data.save_to_disk(
#     "/home/gantumur/Documents/DL/Lab_commonvoice/data/my_voice_dataset_processed")

Saving the dataset (0/1 shards):   0%|          | 0/70 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/30 [00:00<?, ? examples/s]

In [26]:
def prepare_transcribe(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16_000).input_features[0]

    processor.tokenizer.set_prefix_tokens(
        language="Mongolian", task="transcribe")
    batch["labels"] = processor.tokenizer(batch["sentence"]).input_ids
    return batch


def prepare_translation(batch):
    audio = batch["audio"]
    batch["input_features"] = processor.feature_extractor(
        audio["array"], sampling_rate=16_000).input_features[0]

    processor.tokenizer.set_prefix_tokens(
        language="Mongolian", task="translate")
    batch["labels"] = processor.tokenizer(batch["sentence_en"]).input_ids
    return batch

In [27]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, WhisperProcessor

processor = WhisperProcessor.from_pretrained(
    "openai/whisper-tiny", language="Mongolian", task="transcribe")


train_set = data["train"]
test_set = data["validation"]

In [8]:
train_data_dir = "/home/gantumur/Documents/DL/Lab_commonvoice/data/mapped_dataset/custom"

In [29]:
train_transcribe = train_set.map(prepare_transcribe, remove_columns=train_set.column_names, num_proc=1)
test_transcribe = test_set.map(prepare_transcribe, remove_columns=test_set.column_names, num_proc=1)
train_translation = train_set.map(prepare_translation, remove_columns=train_set.column_names, num_proc=1)

# train_transcribe.save_to_disk(f"{train_data_dir}/train_transcribe")
# test_transcribe.save_to_disk(f"{train_data_dir}/test_transcribe")
# train_translation.save_to_disk(f"{train_data_dir}/train_translation")